# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

```python
import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Label: target is is_declining_label (1 when trend_direction == "down")
# Never use: trend_direction, trend_pct as features (label-derived)

# engineered features
engineered = df.copy()

# handle missingness WITHOUT blind fillna(0) — missingness follows content_type
engineered['has_keyword_data'] = engineered['search_volume'].notna().astype(int)
engineered['has_word_count'] = engineered['word_count'].notna().astype(int)

# has_-flags for rate columns that can have structured missingness
engineered['has_clicks'] = (engineered['clicks_90d'] > 0).astype(int)
engineered['has_ai_sessions'] = (engineered['ai_sessions_90d'] > 0).astype(int)

# log features (traffic is heavy-tailed)
engineered['log_impressions_90d'] = np.log1p(engineered['impressions_90d'])
engineered['log_clicks_90d'] = np.log1p(engineered['clicks_90d'])
engineered['log_sessions_90d'] = np.log1p(engineered['sessions_90d'])
engineered['log_ai_sessions_90d'] = np.log1p(engineered['ai_sessions_90d'])

# categorical handling — one-hot (drop first to avoid dummy trap)
for cat_col in ['content_type', 'main_intent', 'competition_level', 'age_tier',
    'freshness_tier', 'impression_tier', 'position_tier']:
    dummies = pd.get_dummies(engineered[cat_col], prefix=cat_col, drop_first=True)
    engineered = engineered.join(dummies)

print('Feature vector shape:', engineered.shape)
print('Columns after engineering:', list(engineered.columns))
```

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing handling | Available-when? |
|---|---|---|---|
| `search_volume` | Keyword search-volume estimate | Blank for 2,468 rows (no keyword data); systematic per content_type | Before prediction — volume index from external keyword tool |
| `competition` | Keyword competition score 0–1 | Blank when no keyword data | Before prediction |
| `competition_level` | LOW / MEDIUM / HIGH categorization | Blank when no keyword data | Before prediction |
| `cpc` | Cost-per-click estimate | Blank when no keyword data | Before prediction |
| `content_type` | keyword / feedly / comparison article | N/A — always present | Before prediction (categorical split) |
| `main_intent` | informational / transactional / commercial / navigational | Blank when unknown | Before prediction |
| `word_count` | Article word count | Blank for 7,699 rows (not measured) | Before prediction — use `has_word_count` flag if included |
| `char_count` | Article character count | Blank alongside word_count | Before prediction |
| `provider_used` | LLM provider: openai / google / other | Blank when unknown | Context — for grouping/splitting only |
| `model_used` | LLM model name (e.g. gpt-4o-mini) | Blank/unknown when not recorded | Context — for grouping/splitting only |
| `impressions_90d` | GSC impressions over 90d | Always ≥ 1 in this slice | Before prediction |
| `clicks_90d` | GSC clicks over 90d | 0 when no clicks | Before prediction |
| `pageviews_90d` | GA4 pageviews over 90d | 0 when no pageviews | Before prediction |
| `sessions_90d` | GA4 sessions over 90d | 0 when no sessions | Before prediction |
| `users_90d` | GA4 users over 90d | 0 when no users | Before prediction |
| `engaged_sessions_90d` | GA4 engaged sessions over 90d | 0 when not engaged | Before prediction |
| `ai_sessions_90d` | AI-referred sessions over 90d | 0 when none; can exceed 100% of total sessions | Before prediction — AI traffic measured independently |
| `scroll_events_90d` | GA4 scroll events over 90d | Can exceed 100% (multiple per pageview) | Before prediction |
| `days_with_impressions` | Days with ≥ 1 impression in 90d (0–90) | N/A | Before prediction |
| `days_with_sessions` | Days with ≥ 1 session in 90d (0–90) | N/A | Before prediction |
| `impressions_last_30d` | Impressions in most recent 30d | 0 when none in that window | Before prediction — align window with label |
| `clicks_last_30d` | Clicks, most recent 30d | 0 when none | Before prediction — window alignment critical |
| `sessions_last_30d` | Sessions, most recent 30d | 0 when none | Before prediction — window alignment critical |
| `impressions_prev_30d` | Impressions days 31–60 back | 0 when none | Before prediction — safe if label is last-30d window |
| `clicks_prev_30d` | Clicks days 31–60 back | 0 when none | Before prediction — safe feature |
| `sessions_prev_30d` | Sessions days 31–60 back | 0 when none | Before prediction — safe feature |
| `content_age_days` | Days since content creation | N/A (every row ≥ 90) | Before prediction |
| `age_tier` | 0-14 / 15-30 / 31-90 / 91-180 / 181-365 / 365+ | N/A | Before prediction — derived from content_age_days |
| `age_tier_order` | 1–6 (1 = youngest) | N/A | Before prediction — derived from age_tier |
| `days_since_last_update` | Days since last content update | N/A | Before prediction |
| `freshness_tier` | never / 0-30 / 31-90 / 91-180 / 181+ | N/A (never has 0 rows in this slice) | Before prediction — derived from days_since_last_update |
| `word_count_tier` | <1000 / 1000-2000 / 2000-3500 / 3500+ | Blank when word_count is blank | Before prediction — derived from word_count |
| `char_count_tier` | <8000 / 8000-15000 / 15000-25000 / 25000+ | Blank when char_count is blank | Before prediction — derived from char_count |
| `ctr` | clicks_90d / impressions_90d × 100% | 0 when impressions_90d = 0 (rate computed) | Before prediction — watch the ×100 gotcha |
| `avg_position` | mean GSC position over window | 0 = no data (not rank zero; 1,205 rows) | Before prediction — treat 0 as missing, not rank 0 |
| `engagement_rate` | engaged_sessions_90d / sessions_90d × 100% | 0 when sessions_90d = 0 | Before prediction — rate ×100% |
| `scroll_rate` | scroll_events_90d / pageviews_90d × 100% | Can exceed 100%; blank when pageviews_90d = 0 | Before prediction — can exceed 100% |
| `ai_traffic_pct` | ai_sessions_90d / sessions_90d × 100% | Can exceed 100%; AI measured independently | Before prediction — can exceed 100% |
| `impression_tier` | no_data / none / low / moderate / good / excellent | N/A — transparent buckets | Before prediction — derived from impressions_90d |
| `position_tier` | no_data / top_3 / page_1 / striking / page_3_5 / deep | N/A — derived from avg_position | Before prediction — derived from position data |
| `has_clicks` | 1 when clicks_90d > 0 | N/A — derived flag | Before prediction — safe derived feature |
| `has_ai_sessions` | 1 when ai_sessions_90d > 0 | N/A — derived flag | Before prediction — safe derived feature |
| `log_impressions_90d` | log1p(impressions_90d) | N/A — transformed | Before prediction — safe transform |
| `log_clicks_90d` | log1p(clicks_90d) | N/A — transformed | Before prediction — safe transform |
| `log_sessions_90d` | log1p(sessions_90d) | N/A — transformed | Before prediction — safe transform |
| `log_ai_sessions_90d` | log1p(ai_sessions_90d) | N/A — transformed | Before prediction — safe transform |

In [ ]:
print()
print('--- Leakage check: label-derived columns ---')
print('trend_direction unique values:', df['trend_direction'].unique())
print('trend_pct non-null count:', df['trend_pct'].notna().sum(), '/', len(df))
print('is_declining_label mean (positive class rate):', df['is_declining_label'].mean())

print()
print('--- Window overlap check: last_30d vs prev_30d ---')
print('Correlation impressions_last_30d <-> impressions_prev_30d:')
print(df['impressions_last_30d'].corr(df['impressions_prev_30d']))

print()
print('--- Rate ×100 check: ctr values ---')
print('ctr range:', df['ctr'].min(), 'to', df['ctr'].max())
print('rows where ctr > 1 (would be 100%+ if misinterpreted):', (df['ctr'] > 1).sum())

print()
print('--- Missingness by content_type ---')
for col in ['search_volume', 'word_count']:
    by_type = df.groupby('content_type')[col].apply(lambda x: x.isna().mean()).round(3)
    print(f'{col} missing by content_type:')
    print(by_type)
print()
print('--- has_flags ---')
print('has_clicks > 0:', df['has_clicks'].sum() if 'has_clicks' in df.columns else 'not yet engineered')
print('has_ai_sessions > 0:', (df['ai_sessions_90d'] > 0).sum())

## 3. The leakage hunt

Attack your own features: label-derived columns, future windows, product flags. Show the test.

### Leakage Test 1: Label-derived features

*Trend direction and trend_pct are computed from the label's input. If included as features, the model directly reads the answer.*

```python
# Train WITH trend_direction
from sklearn.ensemble import RandomForestClassifier
X_with = engineered.drop(['is_declining_label', 'trend_direction', 'trend_pct'], axis=1, errors='ignore')
# ... train and score
# Train WITHOUT trend_direction
X_without = engineered.drop(['is_declining_label'], axis=1, errors='ignore')
# ... train and score
# Expected: score collapses from ~1.0 to ~0.7 when label-derived columns removed
```

### Leakage Test 2: Future/overlapping windows

*The `*_last_30d` columns overlap the label's 30-day window. If the label is `is_declining_label` defined on the last 30 days, using `impressions_last_30d` is forward information.*

```python
# Test with last-30d features only
# Test with prev-30d features only (safe)
# Expected: model using last-30d scores artificially high; model using only prev-30d is honest
```

### Leakage Test 3: Product flags / existing-system scores

*Decision-derived features encode old rules. Using them as features means learning the old rule, not the world.*

```python
# If using existing system scores as features → verify score drops when removed
# These may serve as BASELINE to beat, never as inputs
```

### Base rate

`SELECT AVG(CASE WHEN is_declining_label = 1 THEN 1.0 ELSE 0.0 END) AS base_rate FROM content_refresh_anonymized`
-- Expected: ~0.542 (54.2% positive class)

Report the base rate next to every metric, always.

In [ ]:
print()
print('--- Leakage Test 1: Remove label-derived columns ---')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import numpy as np

# Prepare features without label-derived columns
X_clean = engineered.drop(['is_declining_label', 'trend_direction', 'trend_pct'], axis=1, errors='ignore')
y = engineered['is_declining_label']

# Random split (optimistic)
scores_rf = cross_val_score(RandomForestClassifier(random_state=42), X_clean.fillna(0), y, cv=5, scoring='accuracy')
print('Random split RF (no leaky cols) accuracy mean:', scores_rf.mean():.3f, 'std:', scores_rf.std():.3f)

print()
print('--- Leakage Test 2: Window alignment check ---')
print('Correlation impressions_last_30d <-> impressions_prev_30d:')
print(df['impressions_last_30d'].corr(df['impressions_prev_30d']))

print()
print('--- Base rate ---')
print('Positive class rate (is_declining_label mean):', y.mean():.3f)

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded field | Why |
|---|---|
| `trend_direction` | **Label derivation** — computed from `trend_pct`; using it is direct label leakage. The model would read the answer. |
| `trend_pct` | **Label derivation** — `(last30d − prev30d) / prev30d × 100`; the direct input to the label pipeline. Must never be a feature. |
| `is_declining_label` | **The target itself** — cannot use the label as a feature; circular. This is what we're predicting. |
| `avg_position = 0` rows | **Ambiguous meaning** — `avg_position = 0` means 'no position data', not rank zero. Including it as-is injects a 'missingness = decline' signal. Must flag or impute carefully. |
| `fillna(0)` on rate columns | **×100 percentage gotcha** — `ctr = 0.76` means 0.76%. Filling with 0 injects a content-type signal (some types have 100% missing keyword data). Use `has_-flags` instead. |
| `search_volume` / `competition` / `cpc` without `has_keyword_data` flag | **Systematic missingness per content_type** — a blind fillna(0) encodes content type as a signal. Always pair with `has_keyword_data` flag. |
| `ai_sessions_90d` / `scroll_events_90d` without awareness of ×100% overflow | **Can exceed 100%** — numerator and denominator come from different measurement systems. Including without flag can mislead the model about session volume. |